# Notebook 18 – Transfer Learning Introduction

## 1. What is Transfer Learning?

**Transfer Learning** means using knowledge learned by a model on one task and applying that knowledge to another related task.

Instead of starting with random weights, we start with a model that has already learned useful patterns from a large dataset.

### Simple idea

A pretrained image model may already know how to detect:

- edges
- curves
- textures
- shapes
- patterns
- object parts

We can reuse these learned features and train the final part of the model for our own problem.

### Real-world example

Suppose I want to classify different types of flowers.

Instead of training a CNN from zero, I can use a model pretrained on ImageNet. The early layers can provide useful visual features, while the final classification layer can be changed for flower classes.

### AI/ML usage

Transfer learning is commonly used in:

- image classification
- object detection
- medical image analysis
- face and object recognition
- defect detection
- satellite image analysis
- OCR and document image tasks

### Why it is useful

Transfer learning can:

- reduce training time
- reduce the amount of data needed
- provide better starting weights
- make deep learning practical on smaller datasets


## 2. Pretrained Model

A **pretrained model** is a neural network that has already been trained on a large dataset.

For this notebook I will use **ResNet18**, a well-known convolutional neural network available through TorchVision.

The model was originally trained for image classification.

### Simple example

```text
Large image dataset
        ↓
Pretrained ResNet18
        ↓
Learned visual features
        ↓
My new dataset
        ↓
New classification task
```

### AI/ML usage

Pretrained models are useful when my own dataset is smaller than the dataset normally required to train a deep neural network from scratch.


## 3. Feature Extraction

**Feature extraction** means using the pretrained model to convert an input into useful learned features.

During feature extraction, most of the pretrained model is kept fixed.

Only the new classification part is trained.

### Example

A pretrained CNN may learn:

**Image → edges → textures → shapes → high-level features → class**

I can reuse the learned feature layers and replace the last classifier.

### Real-world example

For a product image classifier:

**Product image → pretrained CNN features → new product categories**

### AI/ML usage

Feature extraction is useful when:

- the new dataset is small
- the new images are reasonably similar to the original training images
- I want faster training


## 4. Fine-Tuning

**Fine-tuning** means allowing some of the pretrained layers to learn again using the new dataset.

Instead of keeping every pretrained layer frozen, I can unfreeze selected deeper layers.

### Example

```text
Early layers       → Frozen
Middle layers      → Frozen
Later layers       → Trainable
New classifier     → Trainable
```

Fine-tuning usually starts after feature extraction.

### Why fine-tune?

The pretrained model already knows general visual patterns. Fine-tuning lets the model adapt some of those patterns to the new problem.

### AI/ML usage

Fine-tuning can help when the new dataset is sufficiently large or visually different from the original training data.


## 5. Frozen Layers

A **frozen layer** is a layer whose parameters are not updated during training.

In PyTorch this is commonly done by setting:

`parameter.requires_grad = False`

### Simple idea

If a layer is frozen:

**Forward pass → yes**

**Backpropagation update → no**

### Real-world example

If I use a pretrained CNN for classifying flowers, I can freeze most of the network and train only the new flower classifier.

### AI/ML usage

Frozen layers reduce the number of parameters that need to be trained, which can reduce:

- training time
- memory usage
- computational cost


## 6. Trainable Layers

A **trainable layer** has parameters that can be updated during training.

For transfer learning, the new classification layer is normally trainable.

During fine-tuning, some deeper pretrained layers can also become trainable.

### Example

```text
Pretrained feature extractor
        ↓
Mostly frozen
        ↓
New classifier
        ↓
Trainable
```

### AI/ML usage

Trainable layers allow the model to adapt its learned representations to the new task.


## 7. Why Transfer Learning is Useful

Training a deep vision model from scratch can require:

- a large dataset
- long training time
- powerful hardware
- careful model design

Transfer learning gives me a strong starting point.

### Advantages

| Benefit | Explanation |
|---|---|
| Faster training | The model already has useful weights |
| Less data | I may not need millions of images |
| Lower cost | Fewer resources are required |
| Good starting point | Learned visual features are reused |
| Easier experimentation | I can quickly test a new classifier |

### Real-world example

A small company wants to classify defective products from photographs. Instead of training a CNN from zero, it can start with a pretrained model and adapt it to the defect categories.


## 8. When Should I Use Transfer Learning?

Transfer learning is a good choice when:

- I have a relatively small dataset
- I am working with images
- a suitable pretrained model exists
- the new task has some similarity to the original task
- I want faster experimentation

### Example

A dataset contains only 3,000 product images.

A pretrained CNN can provide a useful starting point instead of learning every visual feature from random initialization.


## 9. When Can Training From Scratch Be Appropriate?

Training from scratch can make sense when:

- I have a very large dataset
- the domain is very different from common pretrained datasets
- no useful pretrained model exists
- I need complete control over the architecture
- I am doing research on a new architecture

### Example

A specialized scientific imaging problem may contain very different image patterns from ordinary photographs. In such a case, transfer learning still may be tested, but training from scratch can also be considered when enough labeled data and computing resources are available.


## 10. General Transfer Learning Workflow

The workflow I am studying is:

```text
Pretrained Model
       ↓
Freeze or Modify Layers
       ↓
Replace the Classifier
       ↓
Train on New Dataset
       ↓
Evaluate
       ↓
Fine-Tune Selected Layers
       ↓
Evaluate Again
```

This is the basic workflow I can reuse in many computer-vision projects.


# Practical Part

## 11. Load the Provided Dataset

The provided file is the **Online Retail dataset**.

It contains transaction information such as:

- InvoiceNo
- StockCode
- Description
- Quantity
- InvoiceDate
- UnitPrice
- CustomerID
- Country

Because it is tabular data rather than an image dataset, I will use selected numerical retail features to create a compact image-like tensor.

This lets me practice the mechanics of a pretrained vision model while still using the dataset provided for this notebook.


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision import models

df = pd.read_csv("data.csv", encoding="ISO-8859-1")

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [2]:
df.shape

(541909, 8)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


## 12. Basic Data Preparation

I will select four useful numerical features:

- Quantity
- UnitPrice
- CustomerID
- Invoice number converted to a numeric value

The target will be a simple binary label:

**1 = high-value transaction**

**0 = lower-value transaction**

The transaction value is calculated using:

**Quantity × UnitPrice**

This is an educational classification target created from the provided dataset.


In [4]:
data = df[["InvoiceNo", "Quantity", "UnitPrice", "CustomerID"]].copy()

data["InvoiceNo"] = pd.to_numeric(data["InvoiceNo"], errors="coerce")
data["TransactionValue"] = data["Quantity"] * data["UnitPrice"]

data = data.replace([np.inf, -np.inf], np.nan).dropna()

data = data[data["Quantity"] > 0]
data = data[data["UnitPrice"] > 0]

data["Target"] = (data["TransactionValue"] >= data["TransactionValue"].median()).astype(int)

data.head()

,InvoiceNo,Quantity,UnitPrice,CustomerID,TransactionValue,Target
0,536365.0,6,2.55,17850.0,15.30,1
1,536365.0,6,3.39,17850.0,20.34,1
2,536365.0,8,2.75,17850.0,22.00,1
3,536365.0,6,3.39,17850.0,20.34,1
4,536365.0,6,3.39,17850.0,20.34,1


## 13. Create an Image-Like Representation

A pretrained vision model expects image tensors.

Our CSV does not contain images, so I will create a small educational representation from the four selected features.

Each feature is normalized and arranged into a small 2D pattern with three channels.

This is **not a replacement for real product images**. It is only a bridge between the supplied tabular dataset and the requested vision transfer-learning workflow.

For a real project, I would replace this step with actual images.


In [ ]:
features = data[["Quantity", "UnitPrice", "CustomerID", "InvoiceNo"]].values.astype("float32")

minimum = features.min(axis=0)
maximum = features.max(axis=0)

features = (features - minimum) / (maximum - minimum + 1e-8)

features = np.clip(features, 0, 1)

n = len(features)
image_array = np.zeros((n, 3, 32, 32), dtype="float32")

for i in range(n):
    image_array[i, 0] = features[i, 0]
    image_array[i, 1] = features[i, 1]
    image_array[i, 2] = features[i, 2]
    image_array[i, :, :, :4] = features[i, 3]

X = torch.tensor(image_array)
y = torch.tensor(data["Target"].values, dtype=torch.long)

X.shape, y.shape

## 14. Prepare Train and Test Data

I will use a small subset so that the notebook stays lightweight and beginner-friendly.

The split is:

- 80% training
- 20% testing

I will resize the image-like tensors to the input size expected by ResNet18.


In [ ]:
sample_size = min(4000, len(X))

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(X), generator=generator)[:sample_size]

X_small = X[indices]
y_small = y[indices]

split = int(0.8 * sample_size)

X_train = X_small[:split]
y_train = y_small[:split]

X_test = X_small[split:]
y_test = y_small[split:]

X_train = torch.nn.functional.interpolate(X_train, size=(224, 224), mode="bilinear")
X_test = torch.nn.functional.interpolate(X_test, size=(224, 224), mode="bilinear")

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

X_train.shape

## 15. Load a Pretrained ResNet18

TorchVision provides pretrained models that can be reused.

I will load ResNet18 with ImageNet pretrained weights.

The first time this cell is run, TorchVision may need an internet connection to download the weights.

If the weights are already available on the computer, they can be loaded directly.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

model

## 16. Inspect the Model

The final layer of ResNet18 is designed for the original ImageNet classification task.

My task has only two classes, so I need to replace the final classifier.


In [ ]:
model.fc

## 17. Freeze the Pretrained Layers

Now I will freeze the pretrained feature extractor.

Only the new final layer will be trainable.

This is the **feature extraction** stage of transfer learning.


In [ ]:
for parameter in model.parameters():
    parameter.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
total_parameters = sum(parameter.numel() for parameter in model.parameters())

trainable_parameters, total_parameters

### What happened?

Most parameters are now frozen.

The new final layer is trainable.

This means the model is using the pretrained ResNet18 as a feature extractor while learning a new two-class decision layer.


## 18. Train the New Classification Layer

I will use:

- Cross Entropy Loss
- Adam optimizer
- a small learning rate
- a small number of epochs

This is intentionally simple so I can understand the workflow.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

epochs = 2

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print("Epoch:", epoch + 1, "Loss:", running_loss / len(train_loader))

## 19. Evaluate the Model

I will calculate test accuracy to see how the adapted model performs on unseen data.


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = correct / total

print("Test Accuracy:", accuracy)

## 20. Fine-Tuning

Feature extraction is the first stage.

Now I can demonstrate **fine-tuning** by making the last ResNet block trainable.

The idea is:

```text
Early layers → Frozen
Later feature layer → Trainable
New classifier → Trainable
```

This lets the model adjust some higher-level visual features for the new task.


In [ ]:
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

print("Trainable parameters:", trainable_parameters)

## 21. Fine-Tune the Selected Layers

I will use a smaller learning rate for fine-tuning.

A smaller learning rate helps avoid changing the pretrained weights too aggressively.


In [ ]:
optimizer = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, model.parameters()),
    lr=0.0001
)

epochs = 1

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print("Fine-tuning Epoch:", epoch + 1, "Loss:", running_loss / len(train_loader))

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = correct / total

print("Fine-tuned Test Accuracy:", accuracy)

## 22. Frozen vs Trainable Layers

| Concept | Meaning |
|---|---|
| Frozen layer | Parameters are not updated |
| Trainable layer | Parameters are updated during training |
| Feature extraction | Use pretrained layers mainly as fixed features |
| Fine-tuning | Unfreeze selected pretrained layers and train them |
| New classifier | Layer modified for the new task |

### Easy way to remember

**Feature extraction = mostly freeze**

**Fine-tuning = selectively unfreeze**
